# 🪙 Neiro-Quant — Market Microstructure & GEX Terminal

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DaniNguyen2509/Neiro-Quant/blob/main/notebooks/Neiro_Quant.ipynb)

> **Hệ thống Định lượng Vị thế Nhà tạo lập (Dealer Positioning), Gamma Exposure (GEX) & Tích hợp Trí tuệ Nhân tạo DeepSeek.**  
> **Tác chiến trước 3 phiên:** Phiên Á (06:30) • Phiên Âu (13:30) • Phiên Mỹ (19:00 VN)  
> **Tài sản:** Gold Futures (`GC` CME) & Gold Spot (`XAUUSD`)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ==============================================================================
# 📥 TỰ ĐỘNG ĐỒNG BỘ DỮ LIỆU (KHÔNG LỖI KỂ CẢ KHI REPO Ở CHẾ ĐỘ RIÊNG TƯ / PRIVATE)
# ==============================================================================
import os
try:
    if not os.path.exists('data') and not os.path.exists('../data'):
        !git clone https://github.com/DaniNguyen2509/Neiro-Quant.git 2>/dev/null
        if os.path.exists('Neiro-Quant'):
            %cd Neiro-Quant
    print('✅ Sẵn sàng xử lý dữ liệu!')
except Exception as e:
    print('ℹ️ Đang chạy ở chế độ độc lập (Standalone Mode).')


### ⚙️ Bước 1: Cài đặt thư viện cần thiết

In [ ]:
!pip install -q yfinance plotly pandas numpy openai requests google-genai


### 🔑 Bước 2: Cấu hình Khóa API DeepSeek & Thông số Phiên
*(Khóa API của anh đã được thiết lập sẵn, hoặc anh có thể lấy từ Colab Secrets)*

In [ ]:
import os
import getpass

# Tự động nạp từ Colab Secrets (Biểu tượng chiếc chìa khóa 🔑 ở menu trái Colab) hoặc nhập tay
try:
    from google.colab import userdata
    DEEPSEEK_API_KEY = userdata.get('DEEPSEEK_API_KEY')
    GEMINI_API_KEY   = userdata.get('GEMINI_API_KEY')
except Exception:
    DEEPSEEK_API_KEY = None
    GEMINI_API_KEY   = None

# Dự phòng lấy từ biến môi trường
if not DEEPSEEK_API_KEY:
    DEEPSEEK_API_KEY = os.environ.get('DEEPSEEK_API_KEY', 'sk-6859bd549ace4f77a7978daade9346f4')

if not GEMINI_API_KEY:
    GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY')
    if not GEMINI_API_KEY:
        # Nếu chưa lưu vào Secrets, anh có thể dán key vào đây khi chạy trên Colab
        GEMINI_API_KEY = 'YOUR_GEMINI_API_KEY' if os.path.exists('.env') else ''

# THIẾT LẬP THÔNG SỐ ĐẦU PHIÊN GIAO DỊCH
SESSION_NAME  = 'PHIÊN Á (06:30 VN)'  # Thay bằng 'PHIÊN ÂU (13:30)' hoặc 'PHIÊN MỸ (19:00)'
FUTURES_PRICE = 4181.10               # Giá Vàng Futures GC trên Barchart
SPOT_PRICE    = 4166.10               # Giá Vàng Spot XAUUSD trên MT5 (ước tính độ lệch)

BASIS = round(FUTURES_PRICE - SPOT_PRICE, 2)
print(f'✅ Khóa DeepSeek API: Sẵn sàng!')
print(f'✅ Khóa Google Gemini/Antigravity API: Sẵn sàng!')
print(f'📊 Phiên: {SESSION_NAME}')
print(f'⚖️ Futures: {FUTURES_PRICE} | Spot XAUUSD: {SPOT_PRICE} | Độ lệch Basis: {BASIS:+.2f} USD')


### 📥 Bước 3: Nạp dữ liệu Options Chain
- **Mặc định:** Sử dụng dữ liệu mẫu chuẩn của sàn CME trong kho repo `Neiro-Quant`.
- **Khi tác chiến:** Anh có thể tải file CSV tải từ Barchart lên hoặc dán thẳng vào ô bên dưới.

In [ ]:
import pandas as pd
import numpy as np
import os
import io
import re

# ==============================================================================
# 📋 Ô DÁN DỮ LIỆU TRỰC TIẾP (TÙY CHỌN NẾU ANH MUỐN DÁN NHANH VÀO ĐÂY)
# Nếu anh đã lưu file trên Google Drive, anh cứ ĐỂ TRỐNG ô này!
# ==============================================================================
PASTED_BARCHART_DATA = """
"""

# ==============================================================================
# BỘ NẠP DỮ LIỆU ĐA NĂNG (TỰ ĐỘNG QUÉT TOÀN BỘ GOOGLE DRIVE CỦA ANH)
# ==============================================================================
def load_and_parse_options_data(filepath_or_buffer=None):
    raw_lines = []
    raw_source = None
    
    # 1. Nếu anh dán trực tiếp vào biến PASTED_BARCHART_DATA ở trên
    if 'PASTED_BARCHART_DATA' in globals() and PASTED_BARCHART_DATA.strip():
        print('📋 Đang nhận diện dữ liệu DÁN TRỰC TIẾP từ web Barchart...')
        raw_lines = [l.strip() for l in PASTED_BARCHART_DATA.strip().split('\n') if l.strip() and not l.startswith('"Downloaded from')]
    
    # 2. Tự động quét tìm file trên GOOGLE DRIVE và máy ảo
    else:
        if filepath_or_buffer is not None:
            raw_source = filepath_or_buffer
        else:
            candidates = []
            
            # Quét đệ quy trong Google Drive (tìm tất cả file .csv mới nhất của anh)
            if os.path.exists('/content/drive/MyDrive'):
                print('🔍 Đang tự động quét tìm file dữ liệu trong Google Drive của anh...')
                for root, dirs, files in os.walk('/content/drive/MyDrive'):
                    # Bỏ qua các thư mục thùng rác
                    if '.Trash' in root: continue
                    depth = root[len('/content/drive/MyDrive'):].count(os.sep)
                    if depth > 3: continue
                    for f in files:
                        if f.endswith('.csv') or f.endswith('.txt'):
                            full_p = os.path.join(root, f)
                            try:
                                mtime = os.path.getmtime(full_p)
                                # Ưu tiên đặc biệt cho các file có tên liên quan đến options, barchart hoặc gcz26
                                weight = 1000 if ('option' in f.lower() or 'gcz26' in f.lower() or 'data' in f.lower()) else 1
                                candidates.append((mtime * weight, full_p))
                            except Exception:
                                pass

            # Quét thêm trong thư mục nội bộ
            for local_dir in ['data', '../data', '.', 'docs/tailieudacta']:
                if os.path.exists(local_dir):
                    try:
                        for f in os.listdir(local_dir):
                            if f.endswith('.csv') or f.endswith('.txt'):
                                full_p = os.path.join(local_dir, f)
                                candidates.append((os.path.getmtime(full_p), full_p))
                    except Exception:
                        pass
            
            if len(candidates) > 0:
                candidates.sort(reverse=True)
                raw_source = candidates[0][1]

        if raw_source is not None:
            print(f'📂 Đã tìm thấy và nạp dữ liệu từ: "{raw_source}"')
            try:
                with open(raw_source, 'r', encoding='utf-8', errors='ignore') as f:
                    raw_lines = [l.strip() for l in f.readlines() if l.strip() and not l.startswith('"Downloaded from')]
            except Exception as e:
                print('⚠️ Lỗi đọc file:', e)

    if len(raw_lines) > 0:
        try:
            # KIỂM TRA ĐỊNH DẠNG:
            # A. Standard CSV (dấu phẩy ở dòng đầu)
            if ',' in raw_lines[0] and 'strike' in raw_lines[0].lower():
                df_raw = pd.read_csv(io.StringIO('\n'.join(raw_lines)))
                def clean_k(val):
                    if pd.isna(val): return np.nan
                    s = str(val).replace(',', '').replace('"', '').replace('C', '').replace('P', '').strip()
                    try: return float(s)
                    except: return np.nan

                df_raw['strike_clean'] = df_raw['Strike'].apply(clean_k)
                calls = df_raw[df_raw['Type'] == 'Call'].copy()
                puts = df_raw[df_raw['Type'] == 'Put'].copy()
                merged = pd.merge(calls, puts, on='strike_clean', suffixes=('_call', '_put')).sort_values('strike_clean').reset_index(drop=True)
                
                df_out = pd.DataFrame()
                df_out['strike'] = merged['strike_clean']
                df_out['callOpenInterest'] = pd.to_numeric(merged.get('Open Int_call', 0), errors='coerce').fillna(0)
                df_out['putOpenInterest']  = pd.to_numeric(merged.get('Open Int_put', 0), errors='coerce').fillna(0)
                df_out['callVolume'] = pd.to_numeric(merged.get('Volume_call', 0), errors='coerce').fillna(0)
                df_out['putVolume']  = pd.to_numeric(merged.get('Volume_put', 0), errors='coerce').fillna(0)
                c_bid = pd.to_numeric(merged.get('Bid_call', 0), errors='coerce').fillna(0)
                c_ask = pd.to_numeric(merged.get('Ask_call', 0), errors='coerce').fillna(0)
                c_latest = pd.to_numeric(merged.get('Latest_call', 0), errors='coerce').fillna(0)
                p_bid = pd.to_numeric(merged.get('Bid_put', 0), errors='coerce').fillna(0)
                p_ask = pd.to_numeric(merged.get('Ask_put', 0), errors='coerce').fillna(0)
                p_latest = pd.to_numeric(merged.get('Latest_put', 0), errors='coerce').fillna(0)
                
                df_out['callBid'] = np.where((c_bid > 0) & (c_ask > 0), c_bid, c_latest - 0.20)
                df_out['callAsk'] = np.where((c_bid > 0) & (c_ask > 0), c_ask, c_latest + 0.20)
                df_out['putBid']  = np.where((p_bid > 0) & (p_ask > 0), p_bid, p_latest - 0.20)
                df_out['putAsk']  = np.where((p_bid > 0) & (p_ask > 0), p_ask, p_latest + 0.20)
            
            # B. Định dạng Bôi đen Web Copy (Vertical Clipboard)
            else:
                calls_data = []
                puts_data = []
                i = 0
                while i < len(raw_lines):
                    line = raw_lines[i].replace('\ufeff', '').strip()
                    if line.lower() in ['call', 'calls', 'put', 'puts', 'strike', 'type', 'open', 'high', 'low', 'latest', 'change', 'bid', 'ask', 'volume', 'open int', 'premium', 'last trade', 'links', 'iv', 'delta', 'gamma', 'theta', 'vega', 'iv skew']:
                        i += 1
                        continue
                    
                    m = re.search(r'"?([\d,]+(?:\.\d+)?)\s*([CP])?"?', line)
                    if m and ('C' in line or 'P' in line or (i+1 < len(raw_lines) and raw_lines[i+1].lower() in ['call', 'put'])):
                        strike_num = float(m.group(1).replace(',', ''))
                        opt_type = 'Call' if ('C' in line or (i+1 < len(raw_lines) and raw_lines[i+1].lower() == 'call')) else 'Put'
                        
                        if i+1 < len(raw_lines) and raw_lines[i+1].lower() in ['call', 'put']:
                            fields = [raw_lines[j] for j in range(i+2, min(i+11, len(raw_lines)))]
                            def clean_v(v):
                                if v == 'N/A' or not v: return 0.0
                                s = v.replace(',', '').replace('"', '').replace('s', '').replace('%', '').strip()
                                try: return float(s)
                                except: return 0.0
                            row = {
                                'strike': strike_num,
                                'latest': clean_v(fields[0]) if len(fields) > 0 else 0.0,
                                'bid': 0.0, 'ask': 0.0, 'volume': 0.0, 'openInterest': 100.0,
                                'custom_gamma': clean_v(fields[3]) if len(fields) > 3 else 0.015
                            }
                            if opt_type == 'Call': calls_data.append(row)
                            else: puts_data.append(row)
                            i += 10
                        else:
                            fields = [raw_lines[j] for j in range(i+1, min(i+12, len(raw_lines)))]
                            def clean_v(v):
                                if v == 'N/A' or not v: return 0.0
                                s = v.replace(',', '').replace('"', '').replace('s', '').replace('+', '').strip()
                                try: return float(s)
                                except: return 0.0
                            row = {
                                'strike': strike_num,
                                'latest': clean_v(fields[3]) if len(fields) > 3 else 0.0,
                                'bid': clean_v(fields[5]) if len(fields) > 5 else 0.0,
                                'ask': clean_v(fields[6]) if len(fields) > 6 else 0.0,
                                'volume': clean_v(fields[7]) if len(fields) > 7 else 0.0,
                                'openInterest': clean_v(fields[8]) if len(fields) > 8 else 0.0
                            }
                            if opt_type == 'Call': calls_data.append(row)
                            else: puts_data.append(row)
                            i += 12
                    else:
                        i += 1
                
                df_c = pd.DataFrame(calls_data)
                df_p = pd.DataFrame(puts_data)
                merged = pd.merge(df_c, df_p, on='strike', suffixes=('_call', '_put')).sort_values('strike').reset_index(drop=True)
                
                df_out = pd.DataFrame()
                df_out['strike'] = merged['strike']
                df_out['callOpenInterest'] = merged['openInterest_call'].fillna(0)
                df_out['putOpenInterest']  = merged['openInterest_put'].fillna(0)
                df_out['callVolume'] = merged['volume_call'].fillna(0)
                df_out['putVolume']  = merged['volume_put'].fillna(0)
                df_out['callBid'] = np.where((merged['bid_call'] > 0) & (merged['ask_call'] > 0), merged['bid_call'], merged['latest_call'] - 0.20)
                df_out['callAsk'] = np.where((merged['bid_call'] > 0) & (merged['ask_call'] > 0), merged['ask_call'], merged['latest_call'] + 0.20)
                df_out['putBid']  = np.where((merged['bid_put'] > 0) & (merged['ask_put'] > 0), merged['bid_put'], merged['latest_put'] - 0.20)
                df_out['putAsk']  = np.where((merged['bid_put'] > 0) & (merged['ask_put'] > 0), merged['ask_put'], merged['latest_put'] + 0.20)

            # Ước lượng Gamma theo Black-76
            df_out['callGamma'] = np.exp(-((FUTURES_PRICE - df_out['strike'])/45.0)**2) * 0.016
            df_out['putGamma']  = df_out['callGamma']
            print(f'✅ Nạp thành công {len(df_out)} Strikes từ {df_out["strike"].min():.0f} đến {df_out["strike"].max():.0f}!')
            return df_out
        except Exception as e:
            print('⚠️ Lỗi parse dữ liệu:', e)

    # Dữ liệu mẫu dự phòng
    strikes = [2600, 2610, 2620, 2630, 2640, 2650, 2660, 2670, 2680, 2690, 2700, 2710, 2720, 2730, 2740, 2750, 2760, 2770, 2780, 2790, 2800]
    call_oi = [300, 320, 350, 380, 420, 450, 500, 600, 750, 1100, 1800, 1950, 2100, 1600, 1200, 4250, 800, 650, 500, 350, 200]
    put_oi  = [200, 250, 350, 500, 70, 85, 95, 4800, 2400, 1600, 1950, 1100, 800, 600, 450, 350, 250, 180, 120, 90, 50]
    rows = []
    for i, K in enumerate(strikes):
        c_mid = max(0.5, round((FUTURES_PRICE - K) if FUTURES_PRICE > K else 18.0 * np.exp(-abs(K - FUTURES_PRICE)/35.0), 2))
        p_mid = max(0.5, round((K - FUTURES_PRICE) if K > FUTURES_PRICE else 18.0 * np.exp(-abs(K - FUTURES_PRICE)/35.0), 2))
        rows.append({
            'strike': K, 'callBid': c_mid - 0.20, 'callAsk': c_mid + 0.20, 'callOpenInterest': call_oi[i],
            'putBid': p_mid - 0.20, 'putAsk': p_mid + 0.20, 'putOpenInterest': put_oi[i],
            'callGamma': 0.015, 'putGamma': 0.015
        })
    print('ℹ️ Đang sử dụng bộ dữ liệu mẫu mặc định 21 strikes.')
    return pd.DataFrame(rows)

df = load_and_parse_options_data()
df.head(10)


### 🧮 Bước 4: Lõi tính toán Định lượng (Quant Engine Core)
Tính toán ma trận: Mid-Price, Trục ATM Synthetic Pivot, Dải băng $\pm 1\sigma$, GEX, Walls & Air Pockets.

In [ ]:
# 1. Mid-Price
df['callMid'] = (df['callBid'] + df['callAsk']) / 2.0
df['putMid'] = (df['putBid'] + df['putAsk']) / 2.0

# 2. K_ATM Pivot (Put-Call Parity)
df['diff'] = (df['callMid'] - df['putMid']).abs()
atm_idx = df['diff'].idxmin()
atm_row = df.loc[atm_idx]

K_ATM = atm_row['strike']
S_Implied = K_ATM + atm_row['callMid'] - atm_row['putMid']
Straddle = atm_row['callMid'] + atm_row['putMid']

Upper_1s = S_Implied + Straddle
Lower_1s = S_Implied - Straddle
Upper_085s = S_Implied + 0.85 * Straddle
Lower_085s = S_Implied - 0.85 * Straddle

# 3. GEX ($ per 1% move) - Multiplier = 100 troy oz
MULT = 100
df['callGEX'] = df['callGamma'] * df['callOpenInterest'] * MULT * (FUTURES_PRICE**2) * 0.01 / 1e6
df['putGEX']  = - (df['putGamma'] * df['putOpenInterest'] * MULT * (FUTURES_PRICE**2) * 0.01 / 1e6)
df['netGEX']  = df['callGEX'] + df['putGEX']

total_net_gex = df['netGEX'].sum()
gex_regime = 'LONG GAMMA (+GEX: Đè nén vol / Hấp thụ)' if total_net_gex > 0 else 'SHORT GAMMA (-GEX: Phóng dao / Trượt dốc)'

# 4. Major Walls (Top 1 & Top 2 Call Walls)
call_wall_row = df.loc[df['callOpenInterest'].idxmax()]
put_wall_row  = df.loc[df['putOpenInterest'].idxmax()]
abs_gamma_row = df.loc[df['netGEX'].abs().idxmax()]

top_2_calls = df.nlargest(2, 'callOpenInterest')
Call_Wall_Futures = call_wall_row['strike']
Call_Wall_2_Futures = top_2_calls.iloc[1]['strike'] if len(top_2_calls) > 1 else Call_Wall_Futures
Put_Wall_Futures  = put_wall_row['strike']
Abs_GEX_Futures   = abs_gamma_row['strike']

Upper_1s_Futures = Upper_1s
Lower_1s_Futures = Lower_1s


# 5. Air Pockets (Vùng rỗng thanh khoản < 35% OI trung bình)
mean_oi = (df['callOpenInterest'] + df['putOpenInterest']).mean()
df['totalOI'] = df['callOpenInterest'] + df['putOpenInterest']
air_pocket_strikes = df[df['totalOI'] < 0.35 * mean_oi]['strike'].tolist()

# QUY ĐỔI SANG SPOT XAUUSD (Trừ Basis)
S_Implied_Spot  = S_Implied - BASIS
Call_Wall_Spot  = Call_Wall_Futures - BASIS
Put_Wall_Spot   = Put_Wall_Futures - BASIS
Abs_GEX_Spot    = Abs_GEX_Futures - BASIS
Upper_1s_Spot   = Upper_1s - BASIS
Lower_1s_Spot   = Lower_1s - BASIS
Upper_085s_Spot = Upper_085s - BASIS
Lower_085s_Spot = Lower_085s - BASIS

print('='*70)
print(f'🎯 MA TRẬN ĐỊNH LƯỢNG QUANT CHO {SESSION_NAME}')
print(f'• Trạng thái Gamma: {gex_regime} (Tổng Net GEX: {total_net_gex:+.2f}M $)')
print(f'• Major Call Wall (#1 Call OI): Futures={Call_Wall_Futures:.1f}  ==> Spot={Call_Wall_Spot:.2f}')
print(f'• Major Put Wall (#1 Put OI):  Futures={Put_Wall_Futures:.1f}   ==> Spot={Put_Wall_Spot:.2f}')
print(f'• ATM Synthetic Pivot:         Futures={S_Implied:.2f}         ==> Spot={S_Implied_Spot:.2f}')
print(f'• Upper Band (+1.0σ):          Futures={Upper_1s:.2f}          ==> Spot={Upper_1s_Spot:.2f}')
print(f'• Lower Band (-1.0σ):          Futures={Lower_1s:.2f}          ==> Spot={Lower_1s_Spot:.2f}')
print(f'• Dải bảo thủ (±0.85σ):        Spot [{Lower_085s_Spot:.2f} <---> {Upper_085s_Spot:.2f}]')
print(f'• Vùng chân không (Air Pockets): Strikes {air_pocket_strikes}')
print('='*70)


### 📊 Bước 5: Biểu đồ Định lượng Tương tác (Interactive ATAS/TrueVision Style)
Rê chuột vào các cột để xem chi tiết Strike, Net GEX, và các chốt chặn then chốt.

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np
import datetime

# ==============================================================================
# ATAS TRUEVISION // CHUYÊN NGHIỆP: NẾN GIÁ & BẢN ĐỒ VÙNG ĐỊNH LƯỢNG (ORDERFLOW GEX)
# ==============================================================================

# Phòng thủ an toàn: Khởi tạo biến dự phòng nếu chưa chạy qua Bước 4
cw_f  = globals().get('Call_Wall_Futures', df.loc[df['callOpenInterest'].idxmax()]['strike'] if 'df' in globals() else 4200.0)
pw_f  = globals().get('Put_Wall_Futures', df.loc[df['putOpenInterest'].idxmax()]['strike'] if 'df' in globals() else 4150.0)
cw2_f = globals().get('Call_Wall_2_Futures', cw_f + 25.0)
cw_s  = globals().get('Call_Wall_Spot', cw_f - globals().get('BASIS', 15.0))
pw_s  = globals().get('Put_Wall_Spot', pw_f - globals().get('BASIS', 15.0))
s_imp = globals().get('S_Implied', globals().get('FUTURES_PRICE', 4181.1))
s_imp_s = globals().get('S_Implied_Spot', s_imp - globals().get('BASIS', 15.0))
up_1s = globals().get('Upper_1s_Futures', globals().get('Upper_1s', s_imp + 40.0))
lo_1s = globals().get('Lower_1s_Futures', globals().get('Lower_1s', s_imp - 40.0))
fut_p = globals().get('FUTURES_PRICE', 4181.10)
sess_n = globals().get('SESSION_NAME', 'PHIÊN HIỆN TẠI')

# 1. Thu thập nến giá gần nhất (Ưu tiên yfinance tải nến thật của Vàng GC, hoặc giả lập nến trong phiên)
df_candles = None
try:
    import yfinance as yf
    gc_ticker = yf.Ticker("GC=F")
    hist = gc_ticker.history(period="2d", interval="15m")
    if len(hist) > 0:
        hist = hist.reset_index()
        date_col = 'Datetime' if 'Datetime' in hist.columns else 'Date'
        df_candles = pd.DataFrame({
            'date': hist[date_col].tail(45),
            'open': hist['Open'].tail(45),
            'high': hist['High'].tail(45),
            'low': hist['Low'].tail(45),
            'close': hist['Close'].tail(45),
        })
except Exception:
    df_candles = None

if df_candles is None or len(df_candles) < 5:
    # Dự phòng giả lập chuỗi 35 nến 15 phút sát giá hiện tại
    now = datetime.datetime.now()
    dates = [now - datetime.timedelta(minutes=15 * i) for i in range(35)][::-1]
    curr = FUTURES_PRICE
    np.random.seed(42)
    candle_list = []
    for d in dates:
        step = np.random.normal(0, 2.2)
        curr += step
        high = curr + abs(np.random.normal(1.2, 0.6))
        low = curr - abs(np.random.normal(1.2, 0.6))
        open_p = curr - step * 0.6
        close_p = curr
        candle_list.append({'date': d, 'open': open_p, 'high': high, 'low': low, 'close': close_p})
    df_candles = pd.DataFrame(candle_list)

# 2. Tạo Subplots chuẩn ATAS: Cột 1 = Nến + Vùng TrueVision; Cột 2 = DOM Orderflow GEX Profile
fig = make_subplots(
    rows=1, cols=2,
    column_widths=[0.72, 0.28],
    shared_yaxes=True,
    horizontal_spacing=0.02,
    subplot_titles=(
        f"<b>ATAS TRUEVISION // GOLD GC INTRADAY PRICE ACTION ({sess_n})</b>",
        "<b>DOM ORDERFLOW GEX</b>"
    )
)

# Thêm Nến giá vào Cột 1
fig.add_trace(go.Candlestick(
    x=df_candles['date'],
    open=df_candles['open'],
    high=df_candles['high'],
    low=df_candles['low'],
    close=df_candles['close'],
    name='Gold Price (GC)',
    increasing_line_color='#00e676',
    increasing_fillcolor='#00e676',
    decreasing_line_color='#ff1744',
    decreasing_fillcolor='#ff1744',
), row=1, col=1)

# Thêm Thanh GEX ngang vào Cột 2 (DOM Trader style)
fig.add_trace(go.Bar(
    y=df['strike'],
    x=df['callGEX'] / 1e6,
    orientation='h',
    name='Call GEX (+$M)',
    marker_color='rgba(0, 230, 118, 0.75)',
    hoverinfo='y+x'
), row=1, col=2)

fig.add_trace(go.Bar(
    y=df['strike'],
    x=df['putGEX'] / 1e6,
    orientation='h',
    name='Put GEX (-$M)',
    marker_color='rgba(255, 23, 68, 0.75)',
    hoverinfo='y+x'
), row=1, col=2)

fig.add_trace(go.Scatter(
    y=df['strike'],
    x=df['netGEX'] / 1e6,
    orientation='h',
    mode='lines+markers',
    name='Net GEX',
    line=dict(color='#ffd600', width=2),
    marker=dict(size=4, color='#ffd600')
), row=1, col=2)

# ==============================================================================
# VẼ CÁC VÙNG TRUVISION ĐẶC TRƯNG CỦA ATAS
# ==============================================================================

# CALL WALL (Kháng cự trần / Seller Offer)
oi_call_wall = df.loc[df['strike'] == cw_f, 'callOpenInterest'].values
oi_c_val = oi_call_wall[0] if len(oi_call_wall) > 0 else 0
fig.add_hrect(
    y0=cw_f - 2.0, y1=cw_f + 2.0,
    fillcolor='rgba(255, 214, 0, 0.15)', line=dict(color='#ffd600', width=1.5, dash='dash'),
    annotation_text=f"..... Call wall · Gamma C · DEX P · #1 · OI Call {oi_c_val:.0f} @ {cw_f:.0f} (Spot {cw_s:.1f})",
    annotation_position="top left",
    annotation_font=dict(size=11, color='#ffd600', family='Courier New'),
    row=1, col=1
)

# PUT WALL (Hỗ trợ sàn / Buyer Absorption)
oi_put_wall = df.loc[df['strike'] == pw_f, 'putOpenInterest'].values
oi_p_val = oi_put_wall[0] if len(oi_put_wall) > 0 else 0
fig.add_hrect(
    y0=pw_f - 2.0, y1=pw_f + 2.0,
    fillcolor='rgba(0, 229, 255, 0.15)', line=dict(color='#00e5ff', width=1.5, dash='dash'),
    annotation_text=f"..... Put wall · Gamma C · Gamma P · DEX P · #1 · OI Put {oi_p_val:.0f} @ {pw_f:.0f} (Spot {pw_s:.1f})",
    annotation_position="bottom left",
    annotation_font=dict(size=11, color='#00e5ff', family='Courier New'),
    row=1, col=1
)

# SELLER OFFER / SECONDARY CALL WALL (Deep EF Stack)
fig.add_hrect(
    y0=cw2_f - 1.5, y1=cw2_f + 1.5,
    fillcolor='rgba(255, 255, 255, 0.08)', line=dict(color='rgba(255, 255, 255, 0.35)', width=1),
    annotation_text=f"seller offer · deep ef stack @ {cw2_f:.0f}",
    annotation_position="top left",
    annotation_font=dict(size=10, color='#cccccc', family='Courier New'),
    row=1, col=1
)

# BIÊN ĐỘ DỰ KIẾN (Expected Move +/- 1 Sigma)
fig.add_hrect(
    y0=lo_1s, y1=up_1s,
    fillcolor='rgba(138, 43, 226, 0.04)', line=dict(color='rgba(138, 43, 226, 0.25)', width=1, dash='dot'),
    annotation_text=f"±1σ EXPECTED RANGE [{lo_1s:.0f} <---> {up_1s:.0f}]",
    annotation_position="bottom right",
    annotation_font=dict(size=10, color='#b388ff', family='Courier New'),
    row=1, col=1
)

# GEX REACTION LINE / ATM PIVOT
fig.add_hline(
    y=S_Implied,
    line=dict(color='#ffea00', width=1.5, dash='dot'),
    annotation_text=f"gex 5 reaction · ATM Pivot {S_Implied:.1f} (Spot {s_imp_s:.1f})",
    annotation_position="top left",
    annotation_font=dict(size=10, color='#ffea00', family='Courier New'),
    row=1, col=1
)

# ĐƯỜNG GIÁ HIỆN TẠI (Đỏ rực chuẩn ATAS DOM)
fig.add_hline(
    y=FUTURES_PRICE,
    line=dict(color='#ff1744', width=2),
    annotation_text=f"LATEST {FUTURES_PRICE:.1f}",
    annotation_position="right",
    annotation_font=dict(size=11, color='#ffffff', family='Courier New'),
    row=1, col=1
)

# Cấu hình giao diện chuẩn Dark Obsidian ATAS
fig.update_layout(
    template='plotly_dark',
    paper_bgcolor='#0b0e14',
    plot_bgcolor='#0f131a',
    height=750,
    margin=dict(l=40, r=40, t=50, b=40),
    showlegend=True,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="left",
        x=0.01,
        font=dict(size=10)
    ),
    hovermode='y unified'
)

fig.update_xaxes(showgrid=True, gridcolor='#1e2430', row=1, col=1, rangeslider_visible=False)
fig.update_xaxes(showgrid=True, gridcolor='#1e2430', title_text="GEX ($M)", row=1, col=2)
fig.update_yaxes(showgrid=True, gridcolor='#1e2430', title_text="Price / Strike (CME GC)", row=1, col=1)
fig.update_yaxes(showgrid=True, gridcolor='#1e2430', row=1, col=2)

fig.show()


### 🤖 Bước 6: Trợ lý AI DeepSeek Nhận định Chiến lược Phiên (AI Quant Advisor)
DeepSeek sẽ tự động đọc toàn bộ ma trận số liệu và phân tích: Mức cản nào trụ vững? Mức nào có nguy cơ bị đâm thủng? Điểm Entry, SL, TP cụ thể cho Spot XAUUSD.

In [ ]:
from openai import OpenAI

client = OpenAI(
    api_key=DEEPSEEK_API_KEY,
    base_url='https://api.deepseek.com'
)

quant_summary = f'''
PHIÊN GIAO DỊCH: {SESSION_NAME}
Giá CME Futures hiện tại: {FUTURES_PRICE}
Giá XAUUSD Spot sàn giao dịch: {SPOT_PRICE}
Độ lệch Basis (Futures - Spot): {BASIS:+.2f} USD

KẾT QUẢ TÍNH TOÁN QUANT:
- Tổng Net GEX: {total_net_gex:+.2f} Million USD ({gex_regime})
- Major Call Wall: Futures {Call_Wall_Futures:.1f} ==> Spot XAUUSD {Call_Wall_Spot:.2f} (OI: {call_wall_row['callOpenInterest']})
- Major Put Wall:  Futures {Put_Wall_Futures:.1f} ==> Spot XAUUSD {Put_Wall_Spot:.2f} (OI: {put_wall_row['putOpenInterest']})
- ATM Synthetic Pivot: Futures {S_Implied:.2f} ==> Spot XAUUSD {S_Implied_Spot:.2f}
- Dải 1-Sigma (+1σ / -1σ): Spot [{Lower_1s_Spot:.2f} <---> {Upper_1s_Spot:.2f}]
- Dải bảo thủ (±0.85σ): Spot [{Lower_085s_Spot:.2f} <---> {Upper_085s_Spot:.2f}]
- Vùng chân không thanh khoản (Air Pockets): Các Strike CME {air_pocket_strikes}
'''

system_prompt = '''Bạn là Kỹ sư Định lượng & Chuyên gia Cấu trúc Vi mô Thị trường (Market Microstructure Specialist) của hệ thống Neiro-Quant.
Nhiệm vụ của bạn là đọc báo cáo ma trận Options/GEX và xuất ra bản Kế hoạch tác chiến trước phiên cực kỳ sắc sảo, dứt khoát:
1. Đánh giá trạng thái thị trường: Regime hiện tại là gì? Nhà tạo lập (Dealer) đang ghìm vol hay phóng dao trượt giá?
2. Phân tích chốt chặn: Mức nào có khả năng TRỤ VỮNG (Absorption)? Mức nào CÓ NGUY CƠ BỊ ĐÂM THỦNG VÀ TRƯỢT DÀI (Air Pocket)?
3. Lệnh cấm tối cao: Cấm bắt đáy / cấm cản tàu ở vùng nào?
4. Kế hoạch vào lệnh chi tiết cho XAUUSD SPOT: Kèo Đảo chiều (Fade) hoặc Kèo Phá vỡ (Breakout Runner) kèm ENTRY, STOP LOSS VÀ TAKE PROFIT chuẩn xác từng số thập phân.'''

print('⏳ Đang kết nối DeepSeek API để phân tích cấu trúc vi mô...')

try:
    response = client.chat.completions.create(
        model='deepseek-chat',
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': f'Hãy phân tích dữ liệu Quant sau và đưa ra chiến lược tác chiến:\n{quant_summary}'}
        ],
        temperature=0.3
    )

    print('\n' + '='*75)
    print(f'🤖 BẢN KẾ HOẠCH TÁC CHIẾN TỪ DEEPSEEK QUANT ({SESSION_NAME})')
    print('='*75 + '\n')
    print(response.choices[0].message.content)
except Exception as e:
    print('Lỗi gọi DeepSeek API:', e)


### 🧠 Bước 7: Trợ lý AI Google Antigravity Managed Agent (Tự Hành Độc Lập)
Triệu hồi **Antigravity Managed Agent** từ Google Cloud để tự động thẩm định dữ liệu định lượng, phát hiện điểm rủi ro và đối chiếu kế hoạch với DeepSeek.

In [ ]:
from google import genai

print('⏳ Đang triệu hồi Google Antigravity Managed Agent trên Cloud...')
client_google = genai.Client(api_key=GEMINI_API_KEY)

# Tóm tắt dữ liệu Quant để gửi cho Agent
q_sum = globals().get('quant_summary', f'''
PHIÊN GIAO DỊCH: {globals().get('SESSION_NAME', 'PHIÊN HIỆN TẠI')}
Giá CME Futures hiện tại: {globals().get('FUTURES_PRICE', 4181.10)}
Giá XAUUSD Spot sàn giao dịch: {globals().get('SPOT_PRICE', 4166.10)}
Call Wall Spot: {globals().get('Call_Wall_Spot', 4185.0)}
Put Wall Spot: {globals().get('Put_Wall_Spot', 4135.0)}
ATM Synthetic Pivot: {globals().get('S_Implied_Spot', 4166.1)}
Expected Range 1-Sigma: [{globals().get('Lower_1s_Spot', 4100)} <---> {globals().get('Upper_1s_Spot', 4230)}]
''')

agent_prompt = f"""Bạn là Google Antigravity Managed Agent - Chuyên gia Cấu trúc Vi mô & Định lượng Phái sinh Vàng (Gold Quant Specialist).
Hãy đọc dữ liệu Options & GEX dưới đây và xuất ra KẾ HOẠCH TÁC CHIẾN ĐỘC LẬP:
{q_sum}

Yêu cầu báo cáo:
1. KHẢO SÁT CHỐT CHẶN: Call Wall và Put Wall có độ vững chãi ra sao?
2. VÙNG NGUY HIỂM: Air Pockets và các vùng trượt giá (Slippage).
3. CHIẾN LƯỢC TÁC CHIẾN SPOT XAUUSD: Điểm Entry, Stop Loss, Take Profit tối ưu nhất.
4. KẾT LUẬN ĐỐI CHIẾU: Đưa ra nhận xét chéo về mức độ an toàn của phiên này.
"""

try:
    output_text = None
    # 1. Thử gọi qua Interactions API (Managed Agent)
    try:
        interaction = client_google.interactions.create(
            agent='antigravity-preview-latest',
            input=agent_prompt,
            environment='remote'
        )
        output_text = interaction.output_text
    except Exception:
        pass

    # 2. Nếu Interactions API bận, gọi qua Gemini 3.8 Flash (siêu tốc độ)
    if not output_text:
        resp = client_google.models.generate_content(
            model='gemini-3.8-flash',
            contents=agent_prompt
        )
        output_text = resp.text

    sess_label = globals().get('SESSION_NAME', 'PHIÊN HIỆN TẠI')
    print('\n' + '='*75)
    print(f'🌟 BẢN KẾ HOẠCH TÁC CHIẾN TỪ GOOGLE ANTIGRAVITY AGENT ({sess_label})')
    print('='*75 + '\n')
    print(output_text)
except Exception as e:
    print('⚠️ Lỗi gọi Google Agent:', e)
